In [1]:
import os 
import sys

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.append(str(project_root / "src"))

#from by_flight_train_test_split import compute_clustering_representativeness

In [3]:
from by_flight_train_test_split import tag_traintest_seeded_split_subset_WP

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [4]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [5]:
import fiftyone as fo

## list all dataset
print(fo.list_datasets())

# Close any zombie sessions that might be hanging
fo.close_app()

['FLPLAN', 'dugong', 'flplan200', 'flplanpatches']


In [6]:
dataset  = fo.load_dataset("FLPLAN")

In [7]:
dataset.count_values('m_flight')

{'M40': 19,
 'M33': 9,
 'M4': 2,
 'M35': 10,
 'M13': 11,
 'M48': 8,
 'M46': 14,
 'M15': 10}

In [13]:
import random
import numpy as np
from sklearn.model_selection import train_test_split
from fiftyone import ViewField as F


def tag_train_test_seeded_split(
    dataset,
    stratify_by:  str   = "m_flight",
    train_size:   float = 0.85,
    val_size:     float = 0.15,
    test_size:    float = 0.15,
    num_seeds:    int   = 1,
    test_buffer:  float = 0.05,
    verbose:      bool  = True,
):
    """
    Splits a FiftyOne dataset into train / val / test by tagging samples,
    stratified by a categorical field (e.g. flight mission).

    Entire flight missions are kept together — no flight is split across
    train and test.  The approach mirrors the original WP pipeline:

      1. Randomly select flight(s) as test candidates until the test size
         budget is met (with a carry-forward for small flights).
      2. Split remaining flights into train / val using stratified sampling.
      3. Tag all samples with  train_{seed} / val_{seed} / test_{seed}.
         Samples not selected for val are tagged  notin_val_{seed}.
         Samples not selected for test are tagged  notin_TEST_{seed}.

    Parameters
    ----------
    dataset      : FiftyOne dataset
    stratify_by  : sample field used for stratification (e.g. "m_flight")
    train_size   : fraction of non-test samples to use for training
    val_size     : fraction of non-test samples to use for validation
    test_size    : target fraction of full dataset to hold out for test
    num_seeds    : number of independent splits to generate
    test_buffer  : fuzzy tolerance on the test size target
                   (a flight is accepted if its cumulative ratio is within
                    test_size ± test_buffer)
    verbose      : print progress

    Tags written
    ------------
    train_{seed}, val_{seed}, test_{seed},
    notin_val_{seed}, notin_TEST_{seed}
    """

    def _log(msg):
        if verbose:
            print(f"  {msg}")

    # ── Build flight→count dict sorted ascending by size ─────────────────────
    flight_counts = dict(
        sorted(dataset.count_values(stratify_by).items(), key=lambda x: x[1])
    )
    all_flights  = list(flight_counts.keys())
    total_images = dataset.count()

    _log(f"Dataset: {total_images} images  |  "
         f"{len(all_flights)} flights via '{stratify_by}'")
    _log(f"Target split — train:{train_size:.0%}  "
         f"val:{val_size:.0%}  test:{test_size:.0%}")

    used_test_flights = []   # track across seeds to avoid reuse

    for seed in range(num_seeds):
        rng = random.Random(seed)
        _log(f"\n── Seed {seed} ──────────────────────────────")

        # ── Step 1: select test flights ───────────────────────────────────────
        target_test_n   = int(total_images * test_size)
        target_test_max = int(total_images * (test_size + test_buffer))

        # Prioritise unused flights; fall back to full pool if needed
        available = [f for f in all_flights if f not in used_test_flights]
        if not available:
            available = all_flights.copy()

        shuffled = rng.sample(available, len(available))

        test_flights   = []
        test_count     = 0
        remaining_budget = 0

        for flight in shuffled:
            n = flight_counts[flight]
            contribution = n + remaining_budget

            if test_count + contribution <= target_test_max:
                test_flights.append(flight)
                test_count += n
                remaining_budget = 0
            else:
                # flight is too large — carry budget and try next
                remaining_budget += max(0, target_test_n - test_count)

            if test_count >= target_test_n:
                break

        _log(f"Test flights : {test_flights}  "
             f"({test_count} images = {test_count/total_images:.1%})")
        used_test_flights.extend(test_flights)

        # ── Step 2: tag test samples ──────────────────────────────────────────
        for flight in test_flights:
            flight_ids = dataset.match(
                F(stratify_by) == flight
            ).values("id")

            dataset.select(flight_ids).tag_samples(f"test_{seed}")

            _log(f"  Tagged test: {flight}  ({len(flight_ids)} samples)")

        # Tag non-test flight samples as notin_TEST
        non_test_flights = [f for f in all_flights if f not in test_flights]
        for flight in non_test_flights:
            flight_ids = dataset.match(
                F(stratify_by) == flight
            ).values("id")
            dataset.select(flight_ids).tag_samples(f"notin_TEST_{seed}")

        # ── Step 3: train / val split on remaining flights ────────────────────
        trainval_view = dataset.match(
            F(stratify_by).is_in(non_test_flights)
        )
        ids_trainval   = trainval_view.values("id")
        strata_trainval = trainval_view.values(stratify_by)

        train_ids, val_ids = train_test_split(
            ids_trainval,
            test_size=val_size,
            stratify=strata_trainval,
            random_state=seed,
            shuffle=True,
        )

        _log(f"Train: {len(train_ids)}  Val: {len(val_ids)}")

        # ── Step 4: tag train / val ───────────────────────────────────────────
        dataset.select(train_ids).tag_samples(f"train_{seed}")
        dataset.select(val_ids).tag_samples(f"val_{seed}")

        # Tag val samples not selected as notin_val
        # (here all val_ids ARE selected — add notin_val for future subsampling)
        dataset.select(val_ids).tag_samples(f"val_{seed}")

        _log(f"Tagged: train_{seed} ({len(train_ids)})  "
             f"val_{seed} ({len(val_ids)})  "
             f"test_{seed} ({test_count})")

        # ── Summary ───────────────────────────────────────────────────────────
        if verbose:
            print(f"\n  Split summary for seed {seed}:")
            print(f"    train_{seed} : {len(train_ids):>5}  "
                  f"({len(train_ids)/total_images:.1%})")
            print(f"    val_{seed}   : {len(val_ids):>5}  "
                  f"({len(val_ids)/total_images:.1%})")
            print(f"    test_{seed}  : {test_count:>5}  "
                  f"({test_count/total_images:.1%})")

    print(f"\nDone. {num_seeds} seed(s) tagged.")

In [15]:
tag_train_test_seeded_split(
    dataset,
    stratify_by = "m_flight",
    train_size  = 0.85,
    val_size    = 0.15,
    test_size   = 0.15,
    num_seeds   = 3,
    verbose     = True,
)

  Dataset: 83 images  |  8 flights via 'm_flight'
  Target split — train:85%  val:15%  test:15%
  
── Seed 0 ──────────────────────────────
  Test flights : ['M46']  (14 images = 16.9%)
    Tagged test: M46  (14 samples)
  Train: 58  Val: 11
  Tagged: train_0 (58)  val_0 (11)  test_0 (14)

  Split summary for seed 0:
    train_0 :    58  (69.9%)
    val_0   :    11  (13.3%)
    test_0  :    14  (16.9%)
  
── Seed 1 ──────────────────────────────
  Test flights : ['M48', 'M4']  (10 images = 12.0%)
    Tagged test: M48  (8 samples)
    Tagged test: M4  (2 samples)
  Train: 62  Val: 11
  Tagged: train_1 (62)  val_1 (11)  test_1 (10)

  Split summary for seed 1:
    train_1 :    62  (74.7%)
    val_1   :    11  (13.3%)
    test_1  :    10  (12.0%)
  
── Seed 2 ──────────────────────────────
  Test flights : ['M33']  (9 images = 10.8%)
    Tagged test: M33  (9 samples)
  Train: 62  Val: 12
  Tagged: train_2 (62)  val_2 (12)  test_2 (9)

  Split summary for seed 2:
    train_2 :    62  (74.7

In [16]:

print(dataset.count_values("tags"))


{'notin_TEST_2': 74, 'val_2': 12, 'train_2': 62, 'test_1': 10, 'train_0': 58, 'train_1': 62, 'notin_TEST_1': 73, 'notin_TEST_0': 69, 'val_0': 11, 'test_0': 14, 'test_2': 9, 'val_1': 11}


In [21]:
np.unique([k.split('_')[-1] for k in list(dataset.count_values('tags').keys())]).astype('int')

array([0, 1, 2])

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# Load your embeddings
raw = np.array([e for e in dataset.values("full_embeddings") if e is not None])
embs = normalize(raw, norm="l2")

# Compute nearest neighbour L2 distances
knn = NearestNeighbors(n_neighbors=6, metric="euclidean")
knn.fit(embs)
dists, _ = knn.kneighbors(embs)

# Exclude self (distance=0)
nn_dists = dists[:, 1:]

print(f"NN L2 distance stats:")
print(f"  mean  = {nn_dists.mean():.4f}")
print(f"  median= {np.median(nn_dists):.4f}")
print(f"  p10   = {np.percentile(nn_dists, 10):.4f}")
print(f"  p90   = {np.percentile(nn_dists, 90):.4f}")

In [26]:
from active_learning_fiftyone import compute_uniqueness_field_v2, compute_clustering_representativeness

In [28]:
cluster_labels, centroids, embs, ids = compute_clustering_representativeness(
    dataset,
    embeddings_field="full_embeddings",
    cluster_field="cluster_label",
    n_clusters=10,
)

# Step 2 — per-cluster uniqueness
uniq_scores, ids = compute_uniqueness_field_v2(
    dataset,
    embeddings_field="full_embeddings",
    uniqueness_field="uniqueness_score_per_cluster",
    cluster_field="cluster_label",   # ← triggers per-cluster mode
    k=3,
    verbose=True,
)

  Loading embeddings from field 'full_embeddings' ...
 100% |███████████████████| 83/83 [1.2s elapsed, 0s remaining, 81.2 samples/s]         
  Loaded 83 embeddings, dim=1024
  Running KMeans (k=10, n_init=10, seed=42) ...
  Cluster sizes: min=2  mean=8.3  max=23
  Empty clusters: 0 / 10
  Writing field 'cluster_label' to dataset ...
 100% |███████████████████| 83/83 [1.5s elapsed, 0s remaining, 64.4 samples/s]         
bum! Cluster labels saved to 'cluster_label'.
  Loading embeddings from field 'full_embeddings' ...
 100% |███████████████████| 83/83 [1.1s elapsed, 0s remaining, 76.6 samples/s]         
  Loaded 83 embeddings, dim=1024
  Decay: exponential  param=0.5  k=3  weights=[0.607, 0.368, 0.223]
  Loading cluster labels from 'cluster_label' ...
 100% |███████████████████| 83/83 [943.1ms elapsed, 0s remaining, 88.0 samples/s]      
  Found 10 clusters.
DANGER  cluster 0: k_eff=1 >= sqrt(2)=1. Consider reducing k.
    cluster   0: n=   2  k_eff=1  mean=1.0000  max=1.0000
DANGER  

In [29]:
session = fo.launch_app(dataset,
                        auto=False,
                        port=5151)

Session launched. Run `session.show()` to open the App in a cell output.
